# Fine-tuning BONI con Unsloth + Qwen2.5
## Sube tu dataset a Colab o conéctalo desde Google Drive

In [ ]:
# 1. Instalar Unsloth
!pip install unsloth -q
!pip install --upgrade --no-deps xformers trl peft accelerate bitsandbytes -q

In [ ]:
# 2. Cargar modelo Qwen2.5-0.5B con QLoRA
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-0.5B-Instruct",
    max_seq_length=2048,
    dtype=None,  # Auto-detect
    load_in_4bit=True,  # QLoRA: 4-bit para ahorrar VRAM
)

# Target modules for LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    use_rslora=False,
    loftq_config=None,
)

In [ ]:
# 3. Sube tu dataset a Colab o usa Google Drive
from google.colab import files, drive
import json, os

# Opcion A: Subir archivos manualmente
# uploaded = files.upload()

# Opcion B: Montar Google Drive
drive.mount('/content/drive')
dataset_path = "/content/drive/MyDrive/BONI_dataset"  # Crea esta carpeta

# Cargar dataset
train_data = []
with open(f"{dataset_path}/train.jsonl", "r") as f:
    for line in f:
        train_data.append(json.loads(line))

val_data = []
with open(f"{dataset_path}/val.jsonl", "r") as f:
    for line in f:
        val_data.append(json.loads(line))

print(f"Train: {len(train_data)}, Val: {len(val_data)}")

In [ ]:
# 4. Formatear dataset para Unsloth
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template="chatml",  # Qwen2.5 usa formato ChatML
)

def format_conversation(example):
    conv = example["conversations"]
    # Convertir al formato que espera Unsloth
    texts = []
    for msg in conv:
        role = msg["from"]
        content = msg["value"]
        if role == "system":
            texts.append({"role": "system", "content": content})
        elif role == "user":
            texts.append({"role": "user", "content": content})
        elif role == "assistant":
            texts.append({"role": "assistant", "content": content})
    return {"conversations": texts}

train_formatted = [format_conversation(x) for x in train_data]
val_formatted = [format_conversation(x) for x in val_data]

# Tokenizar
def tokenize(example):
    return tokenizer.apply_chat_template(
        example["conversations"],
        tokenize=True,
        add_generation_prompt=False,
        return_dict=True,
    )

train_tokenized = [tokenize(x) for x in train_formatted]
val_tokenized = [tokenize(x) for x in val_formatted]

In [ ]:
# 5. Entrenar con LoRA
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs",
        report_to=[],
        save_steps=10,
        eval_steps=10,
        evaluation_strategy="steps",
    ),
)

# Entrenar
trainer.train()
print("Entrenamiento completado!")

In [ ]:
# 6. Guardar el modelo fine-tuneado
# Guardar LoRA adapter (pequeno ~10MB)
lora_path = "/content/drive/MyDrive/BONI_dataset/boni_lora"
model.save_pretrained(lora_path)
tokenizer.save_pretrained(lora_path)
print(f"LoRA adapter guardado en: {lora_path}")

In [ ]:
# 7. (Opcional) Exportar a GGUF para usar con llama.cpp
# Convierte el modelo base + LoRA en un solo archivo GGUF

model.save_pretrained_gguf(
    "/content/drive/MyDrive/BONI_dataset/boni_qwen2.5_0.5b",
    tokenizer,
    quantization_method="q4_k_m",  # Misma quant que tu modelo local
)
print("GGUF exportado!")
print("Descargalo desde Google Drive y ponlo en:")
print("  C:\Users\nosoy\.leon\local-models\BONI-Qwen2.5-0.5B-Instruct-Q4_K_M.gguf")

In [ ]:
# 8. Probar el modelo fine-tuneado
FastLanguageModel.for_inference(model)

messages = [
    {"role": "system", "content": open(f"{dataset_path}/system_prompt.txt").read()},
    {"role": "user", "content": "hola boni"},
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

outputs = model.generate(input_ids=inputs, max_new_tokens=128, temperature=0.7)
response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
print("BONI:", response)